# v2: class-stratified group split + training + evaluation

Settings: Accelerator **GPU T4 x2**, Internet **On**. Inputs: the `kaggle_inputs_v2` dataset; from the second session on, also the previous version of THIS notebook (Add Input > Your Work > Notebooks).
Secret: `ROBOFLOW_API_KEY`.

**Session plan (one model per session, about 6–8 h each):** session 1 `["YOLOv12"]`, session 2 `["YOLOv11"]`, session 3 `["YOLOv8"]`. The last session also evaluates all models.
Run with **Save Version > Save & Run All**.

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
import glob, os, shutil
RUN_MODELS = ["YOLOv12"]     # change per session
DEVICE = "0,1"
WORK = "/kaggle/working"; os.chdir(WORK)
shutil.copy(glob.glob("/kaggle/input/**/reproduce_results_v2.py", recursive=True)[0], WORK)
manifests = {d: glob.glob(f"/kaggle/input/**/{d}/split_manifest.csv", recursive=True)[0]
             for d in ["Accident", "deteccion_accidentes"]}
print(manifests)

In [ ]:
from roboflow import Roboflow
from kaggle_secrets import UserSecretsClient
rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
SRC = {"deteccion_accidentes": rf.workspace("deteccion-h92uo").project("deteccion_accidentes").version(2)
                                 .download("yolov8", location=f"{WORK}/raw/deteccion_accidentes").location,
       "Accident": rf.workspace("ann-qwhjm").project("accident-yqljb").version(2)
                     .download("yolov8", location=f"{WORK}/raw/Accident").location}

In [ ]:
# Stratified group split. Check that it prints "CLASS COVERAGE: OK" for both datasets.
for d in ["deteccion_accidentes", "Accident"]:
    !python reproduce_results_v2.py split --dataset {d} --src {SRC[d]} --manifest {manifests[d]}

In [ ]:
!python reproduce_results_v2.py train --models {" ".join(RUN_MODELS)} --device {DEVICE}

In [ ]:
# Evaluates every model whose weights exist (this session or previous versions attached as input)
!python reproduce_results_v2.py evaluate
shutil.make_archive(f"{WORK}/paper_results", "zip", f"{WORK}/paper_results")
shutil.rmtree(f"{WORK}/raw", ignore_errors=True); shutil.rmtree(f"{WORK}/data_group_split", ignore_errors=True)
print(os.listdir(WORK))